In [24]:
# Configuration - change YEAR to check each year
FIRE_BASE_DIR = "../../Calabria_dataset/InputReteGood/Target/"
YEAR = "2017"  # change this to: 2008, 2009, ..., 2018

In [25]:
# Imports
import os
import numpy as np
import h5py
import pandas as pd

In [26]:
# Load valid land cells only from zone mapping
df_cell_zones = pd.read_parquet("../1_data/processed/cell_zones.parquet")
valid_cells = set(zip(df_cell_zones["Row"].values, df_cell_zones["Column"].values))
print(f"Total land cells: {len(valid_cells):,}")

Total land cells: 1,483,990


In [27]:
# List all H5 files for the selected year and print date range
year_dir = os.path.join(FIRE_BASE_DIR, YEAR)
fire_files = sorted([f for f in os.listdir(year_dir) if f.endswith(".h5")])

print(f"Year      : {YEAR}")
print(f"First day : {fire_files[0].split('.')[0]}")
print(f"Last day  : {fire_files[-1].split('.')[0]}")
print(f"Total days: {len(fire_files)}")

Year      : 2017
First day : 20170401
Last day  : 20171031
Total days: 159


In [28]:
# Read each day and mark burned land cells in a binary grid
year_grid = None

for fire_file in fire_files:
    print(f"Processing: {fire_file}")
    fire_path = os.path.join(year_dir, fire_file)
    with h5py.File(fire_path, "r") as h5_file:
        values_table = h5_file["values/table"][:]
        attributes_table = h5_file["attributes/table"][:]

    attr_names = [a[0].decode() for a in attributes_table]
    attr_values = [a[1][0] for a in attributes_table]
    attrs = dict(zip(attr_names, attr_values))
    ncols = int(attrs["ncols"])
    nrows = int(attrs["nrows"])

    if year_grid is None:
        year_grid = np.zeros((nrows, ncols), dtype=np.uint8)

    index_values = values_table["index"].astype(int)
    fire_values = values_table["values_block_0"].flatten().astype(int)

    for idx, v in zip(index_values, fire_values):
        row, col = idx // ncols, idx % ncols
        if v == 1 and (row, col) in valid_cells:
            year_grid[row, col] = 1

Processing: 20170401.h5


/tmp/ipykernel_3518005/1544026172.py:21: RuntimeWarning: invalid value encountered in cast
  fire_values = values_table["values_block_0"].flatten().astype(int)


Processing: 20170402.h5
Processing: 20170408.h5
Processing: 20170410.h5
Processing: 20170411.h5
Processing: 20170414.h5
Processing: 20170416.h5
Processing: 20170418.h5
Processing: 20170423.h5
Processing: 20170424.h5
Processing: 20170425.h5
Processing: 20170426.h5
Processing: 20170427.h5
Processing: 20170428.h5
Processing: 20170429.h5
Processing: 20170430.h5
Processing: 20170501.h5
Processing: 20170504.h5
Processing: 20170505.h5
Processing: 20170506.h5
Processing: 20170507.h5
Processing: 20170508.h5
Processing: 20170510.h5
Processing: 20170512.h5
Processing: 20170514.h5
Processing: 20170516.h5
Processing: 20170519.h5
Processing: 20170520.h5
Processing: 20170523.h5
Processing: 20170524.h5
Processing: 20170527.h5
Processing: 20170528.h5
Processing: 20170529.h5
Processing: 20170531.h5
Processing: 20170601.h5
Processing: 20170602.h5
Processing: 20170604.h5
Processing: 20170605.h5
Processing: 20170608.h5
Processing: 20170609.h5
Processing: 20170610.h5
Processing: 20170613.h5
Processing: 2017

In [29]:
# Print total burned area for the selected year
print(f"{YEAR} burned area (land cells only): {int((year_grid > 0).sum())} ha")

2017 burned area (land cells only): 33020 ha
